# 17 — Hard-negative mining on real glitches: a loss oriented to low FAP

**Author:** Giacomo Moretti · *Statistics and Data Analysis*, Università di Milano-Bicocca, A.Y. 2025–2026

**What it does.** Extension 29. Starting from the fresh-injection CNN of notebook 07 (seed 0), fine-tunes it for 8 more epochs in two ways with identical budget: (a) **control**: same fresh injections, noise windows uniformly weighted; (b) **hard-negative mining**: at every epoch the training noise windows are scored by the current network and the top 10 % (the real glitches and the noisiest windows that fool it) are repeated 5 times. Evaluation: test injections of notebooks 05–07, thresholds from the *validation* noise at FAP 10⁻² and 10⁻³, bootstrap errors.

**What it does not do.** One seed per variant and one mining schedule (10 %, ×5): the effect is compared with the bootstrap error of the test set, not with the seed-to-seed scatter of training (≈0.02–0.06 for this CNN, notebook 07). Mining uses only training chunks; the validation noise is never mined.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, time, pickle
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import seq_models as sm
import gwosc_tools as g
g.style()
tf, keras, L = sm._tf()
CH, data, S1, DS = sm.load_data(); TEST = sm.make_test_sets(data, S1); SNRS = sm.SNRS
noise, S1tr = DS["train"]["noise"], S1["train"]
XV, yv = DS["val"]["X"], DS["val"]["yb"]
base = sm.train_variant("cnn_fresh", None, None, seed=0, verbose=False)[0]
print(len(noise), "training noise windows")

16224 training noise windows


In [2]:
EPOCHS, TOPQ, REP = 8, 0.10, 5
def finetune(mining, seed=11):
    m = sm.builder(sm.VARIANTS["cnn_fresh"][0], 0); m.set_weights(base.get_weights())
    m.compile(keras.optimizers.AdamW(3e-4, weight_decay=1e-4), keras.losses.BinaryCrossentropy(from_logits=True))
    r = np.random.default_rng(seed); best, bw, log = np.inf, None, []
    for ep in range(EPOCHS):
        sig, _, _ = sm.make_signals(noise, S1tr, list(g.MORPHS), r)
        nz = noise
        if mining:
            s = m.predict(sm.clip(noise)[..., None], verbose=0, batch_size=1024)[:, 0]
            hard = np.where(s >= np.quantile(s, 1 - TOPQ))[0]; nz = np.concatenate([noise] + [noise[hard]] * (REP - 1))
        X = np.concatenate([sm.clip(nz), sm.clip(sig)])[..., None]; y = np.r_[np.zeros(len(nz)), np.ones(len(sig))].astype(np.float32)
        h = m.fit(X, y, validation_data=(XV, yv), epochs=1, batch_size=128, verbose=0, shuffle=True)
        vl = float(h.history["val_loss"][0]); log.append(vl); print(f"  mining={mining} ep {ep + 1} val {vl:.4f}", flush=True)
    return m, log
fn = os.path.join(g.CACHE, "stats17.pkl")
if os.path.exists(fn):
    ST = pickle.load(open(fn, "rb"))
else:
    ST = {}
    for name, mdl in (("baseline (nb07)", None), ("control +8 epochs", False), ("hard-negative mining", True)):
        t0_ = time.time(); m = base if mdl is None else finetune(mdl)[0]
        P = lambda x, c: m.predict(sm.clip(x)[..., None], verbose=0, batch_size=1024)[:, 0]
        d = {"val": P(DS["val"]["noise"], None), "test": P(data["test"][0], None)}
        for k, (x, c) in TEST.items(): d[k] = P(x, c)
        ST[name] = d; print(name, "%.0f s" % (time.time() - t0_))
    pickle.dump(ST, open(fn, "wb"))

baseline (nb07) 5 s


  mining=False ep 1 val 0.1260


  mining=False ep 2 val 0.1198


  mining=False ep 3 val 0.1407


  mining=False ep 4 val 0.1301


...

  mining=True ep 5 val 0.1167


  mining=True ep 6 val 0.1163


  mining=True ep 7 val 0.1209


  mining=True ep 8 val 0.1162


hard-negative mining 225 s


In [3]:
rows = []
for name, d in ST.items():
    for far in (1e-2, 1e-3):
        for kind in g.MORPHS:
            p, b = g.bootstrap_rho(d["val"], [d[(kind, s)] for s in SNRS], SNRS, far, B=200, seed=9)
            rows.append(dict(model=name, FAP=far, morph=kind, rho50=p[0], err=(g.ci(b[:, 0])[1] - g.ci(b[:, 0])[0]) / 2, obs_test_FAP=float((d["test"] > g.threshold_at_far(d["val"], far)).mean())))
R = pd.DataFrame(rows)
T = R.pivot_table(index="model", columns=["FAP", "morph"], values="rho50", sort=False); T[("mean", 1e-2)] = T[1e-2].mean(axis=1); T[("mean", 1e-3)] = T[1e-3].mean(axis=1)
print(R.groupby(["model", "FAP"]).obs_test_FAP.first().unstack().round(4))
T.round(2)

FAP                    0.001   0.010
model                               
baseline (nb07)       0.0009  0.0039
control +8 epochs     0.0005  0.0052
hard-negative mining  0.0004  0.0035


FAP                   0.01             0.001              mean      
morph                   sg pulse chirp    sg pulse chirp  0.01 0.001
model                                                               
baseline (nb07)       4.69  4.19  4.83  5.84  5.17  5.97  4.57  5.66
control +8 epochs     4.61  3.98  4.54  6.02  5.23  6.15  4.37  5.80
hard-negative mining  4.62  4.07  4.69  5.99  5.32  6.20  4.46  5.84

In [4]:
# paired difference of the mean rho50 (mining - control), bootstrap on the shared validation noise and independent injections
rng = np.random.default_rng(3); diffs = {1e-2: [], 1e-3: []}
for _ in range(200):
    idx = rng.choice(len(ST["control +8 epochs"]["val"]), len(ST["control +8 epochs"]["val"]))
    for far in diffs:
        out = []
        for nm in ("hard-negative mining", "control +8 epochs"):
            d = ST[nm]; thr = g.threshold_at_far(d["val"][idx], far)
            out.append(np.mean([np.nan_to_num(g.snr_level([(rng.choice(d[(k, s)], len(d[(k, s)])) > thr).mean() for s in SNRS], SNRS, .5), nan=16) for k in g.MORPHS]))
        diffs[far].append(out[0] - out[1])
PD = {far: (float(np.mean(v)), *map(float, np.percentile(v, [2.5, 97.5]))) for far, v in diffs.items()}; print("mining - control, mean rho50 (95% CI):", PD)
g.save_json(dict(rows=R.to_dict("records"), mining_minus_control={str(k): v for k, v in PD.items()}, topq=TOPQ, rep=REP, epochs=EPOCHS), "17_summary.json")

mining - control, mean rho50 (95% CI): {0.01: (0.10252258257671933, -0.15266125127143382, 0.33034077235926), 0.001: (0.04240898438660896, -0.21382574064029028, 0.3336237984038995)}


## Output and next step

`results/17_summary.json`. Next: unsupervised glitch families (extension 24) to see *which* real glitches fool the bank and the network.